# Student Policies
Now that you know how model cost, conversion, demand, and optimize price and matching, we can move on the final part of this game, **building your policies!**. Below we have set up skeleton code for you to build your policies. Make when you finish to run the last code cell to upload your policy.

In [ ]:
# Load in all your libraries and packages here!
# NOTE: you are allowed to use packages loaded below ONLY. If you want to use any other packages, please contact the TA.
import pandas as pd
import numpy as np
import itertools
import math
import random
import copy
import time
import pandas as pd
import haversine
import unittest
import pickle
import collections
from utils import * # this imports all helper functions in utils.py

# TODO: Replace the string "TeamName" below with your own team's name in camel case. For e.g. "AwesomeTeam"
# DO NOT RENAME THE FILE, the file name should be "{TEAM_NAME}_Policies.py"
TEAM_NAME = "GRE"


# Pricing Policy

Feel free to add more functions to `StudentPricingPolicy` but **DO NOT** modify the initalization and static methods. Your code will only be graded based on the output of your pricing function.

In [ ]:
class StudentPricingPolicy:
    # DO NOT MODIFY
    def __init__(self, c = 0.70):
        self.c = c

    # DO NOT MODIFY
    @staticmethod
    def get_name():
        return TEAM_NAME

    @staticmethod
    def clamp(x, lo, hi):
        return max(lo, min(hi, x))

    @staticmethod
    def get_origin(r):
        return (r.pickup_lat, r.pickup_lon)

    @staticmethod
    def get_destination(r):
        return (r.dropoff_lat, r.dropoff_lon)

    @staticmethod
    def quick_pair_score(r1, r2):
        """
        Cheap compatibility proxy used before exact routing.
        Lower is better.
        """
        pickup_gap = abs(r1.pickup_area - r2.pickup_area)
        dropoff_gap = abs(r1.dropoff_area - r2.dropoff_area)
        time_gap = abs(r1.arrival_time - r2.arrival_time)
        length_gap = abs(r1.solo_length - r2.solo_length)

        return 2.0 * pickup_gap + 2.0 * dropoff_gap + 0.003 * time_gap + 0.2 * length_gap

    @staticmethod
    def estimate_queue_density(state):
        """
        Queue density in [0, 1].
        """
        return min(1.0, len(state) / 20.0)

    def estimate_matchability(self, rider, state, top_k=6):
        """
        Estimate how matchable this rider is using a few promising candidates.
        """
        if len(state) == 0:
            return 0.0

        candidates = sorted(state, key=lambda w: self.quick_pair_score(rider, w))[:top_k]

        best_saving_ratio = 0.0
        num_good_candidates = 0

        for w in candidates:
            trip_length, shared_length, i_solo_length, j_solo_length, trip_order = populate_shared_ride_lengths(
                self.get_origin(rider), self.get_destination(rider),
                self.get_origin(w), self.get_destination(w)
            )

            solo_sum = rider.solo_length + w.solo_length
            saving = solo_sum - trip_length
            saving_ratio = saving / max(solo_sum, 1e-6)

            if saving_ratio > best_saving_ratio:
                best_saving_ratio = saving_ratio

            if saving_ratio >= 0.12:
                num_good_candidates += 1

        breadth_score = num_good_candidates / max(len(candidates), 1)
        matchability = 0.7 * best_saving_ratio + 0.3 * breadth_score

        return self.clamp(matchability, 0.0, 1.0)

    def estimate_expected_cost_per_mile(self, rider, state):
        """
        Continuous expected cost estimate.

        Idea:
        - Base solo cost is 0.70.
        - Better matchability lowers expected cost.
        - Dense queue lowers expected cost.
        - Long trips and late arrivals slightly increase expected cost.
        """
        queue_density = self.estimate_queue_density(state)
        matchability = self.estimate_matchability(rider, state, top_k=6)
        solo_length = rider.solo_length

        conditional_saving = 0.18 + 0.10 * queue_density

        if solo_length >= 8:
            conditional_saving -= 0.02
        elif solo_length <= 2:
            conditional_saving += 0.02

        COST_PER_MILE = 0.70
        expected_cost = COST_PER_MILE * (1.0 - matchability * conditional_saving)

        if solo_length >= 10:
            expected_cost += 0.015
        elif solo_length <= 2:
            expected_cost -= 0.01

        if rider.arrival_time > 3300:
            expected_cost += 0.015
        elif rider.arrival_time < 600:
            expected_cost -= 0.005

        return self.clamp(expected_cost, 0.45, 0.69)

    def continuous_price_formula(self, rider, state):
        """
        Direct continuous pricing formula.
        This replaces grid search completely.
        """
        queue_density = self.estimate_queue_density(state)
        matchability = self.estimate_matchability(rider, state, top_k=6)
        expected_cost = self.estimate_expected_cost_per_mile(rider, state)
        solo_length = rider.solo_length

        markup = 0.06

        # Easy-to-match riders can be priced lower to stimulate demand
        markup -= 0.10 * matchability

        # Thin queue: lower price to encourage throughput
        markup -= 0.05 * (1.0 - queue_density)

        # Dense queue: can charge a bit more, but not too much
        markup += 0.02 * queue_density

        # Long trips need somewhat more margin
        if solo_length >= 8:
            markup += 0.04
        elif solo_length <= 2:
            markup -= 0.03

        # Mild time effects
        if rider.arrival_time < 600:
            markup -= 0.01
        elif rider.arrival_time > 3300:
            markup += 0.02

        price = expected_cost + markup

        return float(self.clamp(price, 0.35, 0.78))

    def pricing_function(self, state, rider):
        return self.continuous_price_formula(rider, state)

# Matching Policy

Again, feel free to add more functions to `StudentMatchingPolicy` but **DO NOT** modify the initalization and static methods. Your code will only be graded based on the output of your matching function.

In [ ]:

class StudentMatchingPolicy:
    # DO NOT MODIFY
    def __init__(self, c = 0.70):
        self.c = c

    # DO NOT MODIFY
    @staticmethod
    def get_name():
        return TEAM_NAME

    @staticmethod
    def clamp(x, lo, hi):
        return max(lo, min(hi, x))

    @staticmethod
    def get_origin(r):
        return (r.pickup_lat, r.pickup_lon)

    @staticmethod
    def get_destination(r):
        return (r.dropoff_lat, r.dropoff_lon)

    @staticmethod
    def quick_pair_score(r1, r2):
        """
        Cheap compatibility proxy used before exact routing.
        Lower is better.
        """
        pickup_gap = abs(r1.pickup_area - r2.pickup_area)
        dropoff_gap = abs(r1.dropoff_area - r2.dropoff_area)
        time_gap = abs(r1.arrival_time - r2.arrival_time)
        length_gap = abs(r1.solo_length - r2.solo_length)

        return 2.0 * pickup_gap + 2.0 * dropoff_gap + 0.003 * time_gap + 0.2 * length_gap

    def candidate_exact_match_score(self, rider, waiting_rider):
        """
        Exact pair evaluation using the provided helper.
        """
        trip_length, shared_length, i_solo_length, j_solo_length, trip_order = populate_shared_ride_lengths(
            self.get_origin(rider), self.get_destination(rider),
            self.get_origin(waiting_rider), self.get_destination(waiting_rider)
        )

        solo_sum = rider.solo_length + waiting_rider.solo_length
        saving = solo_sum - trip_length
        saving_ratio = saving / max(solo_sum, 1e-6)
        shared_ratio = shared_length / max(trip_length, 1e-6)
        detour_penalty = (i_solo_length + j_solo_length) / max(trip_length, 1e-6)

        score = 2.2 * saving_ratio + 0.35 * shared_ratio + 0.18 * saving - 0.18 * detour_penalty

        return {
            "saving": saving,
            "saving_ratio": saving_ratio,
            "shared_ratio": shared_ratio,
            "detour_penalty": detour_penalty,
            "score": score,
        }

    # def matching_function(self, state, rider):
    #     if len(state) == 0:
    #         return None

    #     max_candidates = 8
    #     base_min_saving_ratio = 0.12
    #     base_min_absolute_saving = 0.35
    #     large_queue_threshold = 8

    #     # Stage 1: cheap shortlist
    #     candidates = sorted(state, key=lambda w: self.quick_pair_score(rider, w))[:max_candidates]

    #     min_saving_ratio = base_min_saving_ratio
    #     min_absolute_saving = base_min_absolute_saving

    #     # Large queue => match more aggressively
    #     if len(state) >= large_queue_threshold:
    #         min_saving_ratio -= 0.02
    #         min_absolute_saving -= 0.08

    #     # Late arrival => waiting is less attractive
    #     if rider.arrival_time > 3300:
    #         min_saving_ratio -= 0.015
    #         min_absolute_saving -= 0.05

    #     min_saving_ratio = max(0.05, min_saving_ratio)
    #     min_absolute_saving = max(0.12, min_absolute_saving)

    #     best_candidate = None
    #     best_score = -1e18

    #     # Stage 2: exact evaluation
    #     for w in candidates:
    #         stats = self.candidate_exact_match_score(rider, w)

    #         saving = stats["saving"]
    #         saving_ratio = stats["saving_ratio"]
    #         score = stats["score"]

    #         if saving_ratio >= min_saving_ratio and saving >= min_absolute_saving:
    #             if score > best_score:
    #                 best_score = score
    #                 best_candidate = w

    #     return best_candidate
    def matching_function(self, state, rider):
        if len(state) == 0:
            return None

        # More aggressive than before
        # max_candidates = 12
        # base_min_saving_ratio = 0.09
        # base_min_absolute_saving = 0.22
        # large_queue_threshold = 6
        max_candidates = 10
        base_min_saving_ratio = 0.10
        base_min_absolute_saving = 0.28
        large_queue_threshold = 6

        # Stage 1: cheap shortlist
        candidates = sorted(state, key=lambda w: self.quick_pair_score(rider, w))[:max_candidates]

        min_saving_ratio = base_min_saving_ratio
        min_absolute_saving = base_min_absolute_saving

        # Large queue => match more aggressively
        if len(state) >= large_queue_threshold:
            min_saving_ratio -= 0.02
            min_absolute_saving -= 0.06

        # Very large queue => even more aggressive
        if len(state) >= 15:
            min_saving_ratio -= 0.02
            min_absolute_saving -= 0.05

        # Late arrival => waiting is less attractive, so match more aggressively
        if rider.arrival_time > 3000:
            min_saving_ratio -= 0.02
            min_absolute_saving -= 0.05

        if rider.arrival_time > 3300:
            min_saving_ratio -= 0.02
            min_absolute_saving -= 0.05

        min_saving_ratio = max(0.03, min_saving_ratio)
        min_absolute_saving = max(0.08, min_absolute_saving)

        best_candidate = None
        best_score = -1e18

        # Stage 2: exact evaluation
        for w in candidates:
            stats = self.candidate_exact_match_score(rider, w)

            saving = stats["saving"]
            saving_ratio = stats["saving_ratio"]
            shared_ratio = stats["shared_ratio"]
            detour_penalty = stats["detour_penalty"]
            score = stats["score"]

            # Bonus for routes with strong shared structure
            adjusted_score = score
            if shared_ratio >= 0.20:
                adjusted_score += 0.05
            if detour_penalty <= 0.55:
                adjusted_score += 0.05

            # Aggressive matching criterion
            if saving_ratio >= min_saving_ratio and saving >= min_absolute_saving:
                if adjusted_score > best_score:
                    best_score = adjusted_score
                    best_candidate = w

        return best_candidate

# Testing your Code

Use the function below to test your policies. This output will only tell you the price and matching result from your policies.

The test examples consist of 4 states and 1 incoming rider:
- Four States: These represent states with 0, 8, 35, and 77 waiting requests, respectively. Each state is created by aggregating all arriving riders within random time windows of 0 seconds, 15 seconds, 1 minute, and 2 minutes from the training data.
- Incoming Rider: This rider is randomly sampled from the training data.

In [4]:
from utils import test_policies
test_policies(StudentPricingPolicy, StudentMatchingPolicy)


=============== Pricing at State 0 (0 waiting requests) ===============
Pricing decision: 0.70000.
Execution time of the pricing function is 0.00000 seconds.

=============== Matching at State 0 (0 waiting requests) ===============
Matching decision: do not match.
Execution time of the matching function is 0.00000 seconds.

=============== Pricing at State 1 (8 waiting requests) ===============
Pricing decision: 0.64604.
Execution time of the pricing function is 0.00200 seconds.

=============== Matching at State 1 (8 waiting requests) ===============
Matching decision: match the incoming rider with a waiting request.
Execution time of the matching function is 0.00000 seconds.

=============== Pricing at State 2 (35 waiting requests) ===============
Pricing decision: 0.66078.
Execution time of the pricing function is 0.00100 seconds.

=============== Matching at State 2 (35 waiting requests) ===============
Matching decision: match the incoming rider with a waiting request.
Execution 

## Congrats! You finished!

Once you have completed this notebook, you can run the following command to export your notebook to a python script. Make sure to submit only `{TEAM_NAME}.py` file. Please do not edit anything!

In [ ]:
# # DO NOT MODIFY
# from utils import export_notebook
# # export the notebook as "{TEAM_NAME}.py"
# export_notebook(TEAM_NAME)

[NbConvertApp] Converting notebook student_policies.ipynb to script


Converted student_policies.ipynb to TeamName.py successfully!


[NbConvertApp] Writing 3740 bytes to student_policies.py
